In [0]:
import json
import importlib.util
import uuid
from spark_schemas import *
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

dbutils.widgets.text("config_path", "config.json")
config_path = Path(dbutils.widgets.get("config_path")).expanduser().resolve()
script_path = config_path.parent / "extract.py"
if not config_path.is_file() or not script_path.is_file():
    raise FileNotFoundError(f"Expected config.json and extract.py together: {config_path.parent}")
spec = importlib.util.spec_from_file_location("citibike_extractor", script_path)
extractor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(extractor)
config = extractor.load_config(config_path)
batch_id = str(uuid.uuid4())
print("batch_id:", batch_id, "mode:", config["mode"],
      "run_type:", config.get("run_type", "manual"), "dry_run:", config["dry_run"])



In [0]:
discovered = extractor.discover_archives(config)
print(f"Discovered {len(discovered)} ZIP(s) matching the configured month range")
if discovered:
    display(spark.createDataFrame(discovered).select("data_month", "source_zip_name", "zip_key", "zip_size_bytes"))


In [0]:
events_table = config["audit_events_table"]
files_table = config["audit_files_table"]

# Existing audit tables are used as-is. If absent, create empty Delta tables
# with the agreed schema; existing incompatible schemas will fail explicitly.
if not spark.catalog.tableExists(events_table):
    spark.createDataFrame([], EVENT_SCHEMA).write.format("delta").saveAsTable(events_table)
if not spark.catalog.tableExists(files_table):
    spark.createDataFrame([], FILE_SCHEMA).write.format("delta").saveAsTable(files_table)

discovered_df = spark.createDataFrame(discovered, DISCOVERY_SCHEMA)

succeeded = (spark.table(events_table)
    .filter((F.col("stage") == "UNZIP") & (F.col("status") == "SUCCESS"))
    .select("zip_key").distinct())

if config.get("force", False):
    approved_df = discovered_df
else:
    approved_df = discovered_df.join(succeeded, "zip_key", "left_anti")

if config["mode"] == "incremental":
    # Find only the earliest pending month; no full audit collect.
    next_month_row = approved_df.agg(F.min("data_month").alias("next_month")).first()
    next_month = next_month_row["next_month"]
    print("Next pending month:", next_month or "none")
    approved_df = approved_df.filter(F.col("data_month") == next_month) if next_month else approved_df.limit(0)

limit = int(config.get("max_files_per_run", 24))
if approved_df.limit(limit + 1).count() > limit:
    raise ValueError(f"More than {limit} ZIPs approved. Narrow config month range.")

approved_zips = [r.asDict() for r in approved_df.orderBy("data_month", "source_zip_name").collect()]
print(f"Approved {len(approved_zips)} / {len(discovered)} ZIPs")
if approved_zips:
    display(spark.createDataFrame(approved_zips, DISCOVERY_SCHEMA))


In [0]:
result = extractor.extract_archives(config, approved_zips, batch_id)

# For historical mode, audit all skips in the selected range.
# For incremental mode, audit skips only for the month being processed.
approved_keys = {row["zip_key"] for row in approved_zips}
if config["mode"] == "incremental":
    active_months = {row["data_month"] for row in approved_zips}
    skipped = [item for item in discovered
               if item["data_month"] in active_months and item["zip_key"] not in approved_keys]
else:
    skipped = [item for item in discovered if item["zip_key"] not in approved_keys]

for item in skipped:
    result["audit_events"].append(extractor.audit_event(
        batch_id, item, "ZIP_CHECK", "SKIPPED", extractor.now(),
        size_bytes=item["zip_size_bytes"]
    ))

print("Extraction status:", result["status"])
print("Audit events:", len(result["audit_events"]), "CSV records:", len(result["audit_files"]))
print(json.dumps(result, indent=2))


In [0]:
if config["dry_run"]:
    print("DRY_RUN: no files downloaded and no audit tables updated")
else:
    if result["audit_files"]:
        (spark.createDataFrame(result["audit_files"], FILE_SCHEMA)
            .write.format("delta").mode("append").saveAsTable(files_table))
    if result["audit_events"]:
        (spark.createDataFrame(result["audit_events"], EVENT_SCHEMA)
            .write.format("delta").mode("append").saveAsTable(events_table))
    print("Audit records appended to Delta")

if result["status"] == "FAILED":
    raise RuntimeError(f"Extraction failed after audit JSON processing: {result['error_message']}")


In [0]:
if not config["dry_run"]:
    display(spark.table(events_table).filter(F.col("batch_id") == batch_id)
        .orderBy("started_at", "source_zip_name", "stage"))
    display(spark.table(files_table).filter(F.col("batch_id") == batch_id)
        .orderBy("source_zip_name", "csv_name"))
